# Circuit Detective Phase 1 Pilot

This notebook is the first training scaffold for the L1 induction-localization pilot: `Qwen/Qwen3.5-2B` + TRL `GRPOTrainer` + Unsloth 4-bit LoRA on the Phase 1 Circuit Detective environment.

It assumes you are running from a clone of this repo. The environment itself stays OpenEnv-valid, but the trainer uses a direct Python wrapper with explicit public tool methods because that matches TRL's `environment_factory` contract and removes an unnecessary websocket failure surface for the first pilot.

## Install

The main notebook runtime keeps `openenv-core` and trainer dependencies. TransformerLens is installed into a sidecar `.venv-tlens` because its current dependency constraints conflict with the OpenEnv stack.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

print("repo root:", Path.cwd())

# Run in a fresh notebook runtime.
%pip install -q "openenv-core[core]==0.2.3" "trl==1.1.0" "unsloth" "datasets>=3.0.0" "matplotlib>=3.8" "jmespath" "peft" "bitsandbytes"
%pip install -q -e .
!python -m venv .venv-tlens
!./.venv-tlens/bin/python -m pip install -q --upgrade pip
!HF_HUB_DISABLE_XET=1 ./.venv-tlens/bin/pip install -q "transformer-lens==2.18.0"

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported

PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOConfig, GRPOTrainer

from circuit_detective.phase1_grpo import (
    CircuitDetectiveToolEnv,
    build_phase1_dataset,
    reward_func,
)

model_name = "Qwen/Qwen3.5-2B"
max_seq_length = 1024
lora_rank = 8
repeats_per_prompt = 16  # Smoke-test default. Raise to 64 for a longer pilot.
output_dir = "outputs/phase1_qwen35_2b_grpo"

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
train_dataset = build_phase1_dataset(repeats_per_prompt=repeats_per_prompt)
print(f"rows={len(train_dataset)}")
train_dataset[0]

In [ ]:
# Sanity-check one live environment episode before launching training.
import json

debug_env = CircuitDetectiveToolEnv()
print(debug_env.reset())
scores_payload = json.loads(debug_env.inspect_induction_scores(top_k=3))
print(scores_payload)
top_head = scores_payload["result"]["scores"][0]["head_id"]
print(debug_env.submit_circuit(heads=[top_head]))
debug_env.close()

In [ ]:
training_args = GRPOConfig(
    output_dir=output_dir,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=1,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=4,
    generation_batch_size=4,
    max_completion_length=384,
    max_steps=100,
    save_steps=50,
    max_grad_norm=0.1,
    report_to="none",
    log_completions=True,
    chat_template_kwargs={"enable_thinking": False},
    use_vllm=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_func],
    args=training_args,
    train_dataset=train_dataset,
    environment_factory=CircuitDetectiveToolEnv,
)

In [ ]:
trainer.train()
trainer.save_model(f"{output_dir}/final_adapter")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

artifacts_dir = Path("artifacts/phase1")
artifacts_dir.mkdir(parents=True, exist_ok=True)
log_history = trainer.state.log_history
available_keys = sorted({key for entry in log_history for key in entry})
print("available log keys:", available_keys)

def save_curve(key: str, path: Path, title: str) -> None:
    xs = [entry["step"] for entry in log_history if "step" in entry and key in entry]
    ys = [entry[key] for entry in log_history if "step" in entry and key in entry]
    if not xs:
        raise ValueError(f"No values found for {key!r}. Available keys: {available_keys}")
    plt.figure(figsize=(6, 4))
    plt.plot(xs, ys)
    plt.xlabel("step")
    plt.ylabel(key)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

save_curve("loss", artifacts_dir / "phase1_loss_curve.png", "Phase 1 Loss")

reward_key = next(
    (
        key
        for key in [
            "reward",
            "rewards",
            "mean_reward",
            "objective/reward",
            "reward/mean",
        ]
        if any(key in entry for entry in log_history)
    ),
    None,
)

if reward_key is None:
    print("No reward-like key found automatically. Inspect available_keys above and rerun save_curve with the correct key.")
else:
    save_curve(
        reward_key,
        artifacts_dir / "phase1_reward_curve.png",
        f"Phase 1 Reward ({reward_key})",
    )
    print("saved:", artifacts_dir / "phase1_reward_curve.png")

print("saved:", artifacts_dir / "phase1_loss_curve.png")